In [1]:
import json
from pathlib import Path

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def dump_json(obj, path: str | Path):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


In [2]:
from __future__ import annotations
from typing import Tuple, List
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM_INFER_TYPES = ("int64", "float64", "int32", "float32")

def split_features_target(df: pd.DataFrame, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    y = df[target_col].astype(str)
    X = df.drop(columns=[target_col])
    return X, y

def detect_cols(X: pd.DataFrame) -> Tuple[List[str], List[str]]:
    num_cols, cat_cols = [], []
    for c in X.columns:
        if X[c].dtype.name in NUM_INFER_TYPES:
            num_cols.append(c)
        else:
            cat_cols.append(c)
    return num_cols, cat_cols

def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = detect_cols(X)
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False))  # robusto a esparsidad
    ])
    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=True))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop"
    )
    return preprocessor

def infer_feature_names(preprocessor: ColumnTransformer) -> List[str]:
    # Extrae nombres expandido tras One-Hot
    feature_names = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(list(cols))
        elif name == "cat":
            ohe = trans.named_steps["onehot"]
            ohe_names = ohe.get_feature_names_out(cols)
            feature_names.extend(ohe_names.tolist())
    return feature_names


In [3]:
from __future__ import annotations
from typing import Dict, Any
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

def build_estimator(name: str, class_weight="balanced", n_jobs=-1):
    if name == "logreg":
        return LogisticRegression(
            max_iter=2000,
            class_weight=class_weight,
            n_jobs=n_jobs,
            solver="lbfgs"
        )
    elif name == "rf":
        return RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            class_weight=class_weight,
            n_jobs=n_jobs,
            random_state=42
        )
    else:
        raise ValueError(f"Estimator '{name}' no soportado")

def maybe_calibrate(clf, method="sigmoid", cv=3):
    # Opcional, si quieres calibración: descomenta en entrenamiento
    return CalibratedClassifierCV(base_estimator=clf, method=method, cv=cv)


In [4]:
from __future__ import annotations
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, average_precision_score, precision_recall_curve
)

def compute_metrics(y_true, proba_beta, labels_order=("Alpha", "Beta"), threshold=0.5):
    y_pred = np.where(proba_beta >= threshold, labels_order[1], labels_order[0])

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_beta": f1_score(y_true, y_pred, pos_label=labels_order[1]),
        "precision_beta": precision_score(y_true, y_pred, pos_label=labels_order[1]),
        "recall_beta": recall_score(y_true, y_pred, pos_label=labels_order[1]),
        "pr_auc_beta": average_precision_score((y_true==labels_order[1]).astype(int), proba_beta),
    }
    cm = confusion_matrix(y_true, y_pred, labels=list(labels_order))
    cm_norm = confusion_matrix(y_true, y_pred, labels=list(labels_order), normalize="true")
    return metrics, cm, cm_norm

def select_threshold(y_true, proba_beta, labels_order=("Alpha", "Beta"), mode="f1_beta", min_precision_beta=None):
    prec, rec, thr = precision_recall_curve((y_true==labels_order[1]).astype(int), proba_beta)
    # El vector 'thr' tiene len = len(prec)-1
    f1 = 2*prec[:-1]*rec[:-1] / (prec[:-1]+rec[:-1] + 1e-12)

    if mode == "f1_beta":
        idx = np.nanargmax(f1)
    elif mode == "recall_beta":
        idx = np.nanargmax(rec[:-1])
    elif mode == "custom" and min_precision_beta is not None:
        valid = np.where(prec[:-1] >= min_precision_beta)[0]
        idx = valid[np.argmax(f1[valid])] if len(valid) else np.nanargmax(f1)
    else:
        idx = np.nanargmax(f1)

    best_thr = float(thr[idx]) if idx < len(thr) else 0.5
    return best_thr, {"precision": float(prec[idx]), "recall": float(rec[idx]), "f1_beta_at_thr": float(f1[idx])}

def plot_confusion(cm, labels, path_png, normalize=False, title=None):
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    sns.heatmap(cm, annot=True, fmt=".3f" if normalize else "d",
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Verdadero")
    ax.set_title(title or ("Matriz de confusión (normalizada)" if normalize else "Matriz de confusión"))
    Path(path_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path_png, dpi=160)
    plt.close(fig)

def plot_pr_curve(y_true, proba_beta, path_png, positive_label="Beta"):
    from sklearn.metrics import precision_recall_curve, average_precision_score
    y = (y_true == positive_label).astype(int)
    prec, rec, _ = precision_recall_curve(y, proba_beta)
    ap = average_precision_score(y, proba_beta)
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ax.plot(rec, prec, lw=2)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"PR curve {positive_label} (AP={ap:.3f})")
    Path(path_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path_png, dpi=160)
    plt.close(fig)
